In [808]:
import pandas as pd
import numpy as np


In [810]:
import pandas as pd
df = pd.read_csv("data/csv/dataset_2_final.csv")
df.head()
df.shape

(57915, 11)

In [812]:
df.isna().sum()


episode_id            0
patient_id            0
start_date            0
end_date              0
encounter_type        0
length_of_stay        0
disease           28287
procedure          5803
cost              57915
gender                0
birth_date            0
dtype: int64

In [814]:
date_cols = ["start_date", "end_date", "birth_date"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)


In [815]:
df[date_cols].dtypes


start_date    datetime64[ns, UTC]
end_date      datetime64[ns, UTC]
birth_date    datetime64[ns, UTC]
dtype: object

In [816]:
df["computed_los"] = (df["end_date"] - df["start_date"]).dt.days


In [820]:
los_mismatch = (df["length_of_stay"] != df["computed_los"]).sum()
los_mismatch


0

In [822]:
df["length_of_stay"] = df["computed_los"]
df.drop(columns="computed_los", inplace=True)


In [824]:
df.shape

(57915, 11)

In [826]:
df["procedure"] = df["procedure"].fillna("NO_PROCEDURE")


In [828]:
df["procedure"].value_counts().head()


procedure
NO_PROCEDURE                                              5803
Depression screening (procedure)                          5105
Assessment of health and social care needs (procedure)    3075
Renal dialysis (procedure)                                2990
Assessment of substance use (procedure)                   2165
Name: count, dtype: int64

In [830]:
df.loc[df["procedure"] == "NO_PROCEDURE", "cost"] = 0


In [832]:
invalid_cost_rows = df[(df["procedure"] != "NO_PROCEDURE") & (df["cost"].isna())]
len(invalid_cost_rows)


52112

In [834]:
cat_cols = ["disease", "procedure", "encounter_type", "gender"]

for col in cat_cols:
    df[col] = df[col].str.strip().str.lower()


In [836]:
df["age_at_episode"] = (
    (df["start_date"] - df["birth_date"]).dt.days // 365
)


In [838]:
df.drop(columns="birth_date", inplace=True)


In [840]:
episode_disease_check = (
    df.groupby("episode_id")["disease"].nunique().max()
)
episode_disease_check


1

In [842]:
gender_conflicts = (
    df.groupby("patient_id")["gender"].nunique()
)
gender_conflicts[gender_conflicts > 1].count()


0

In [844]:
df["gender"] = df["gender"].map({"male": 0, "female": 1})


In [846]:
df = pd.get_dummies(df, columns=["encounter_type"], drop_first=True)


In [848]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57915 entries, 0 to 57914
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   episode_id           57915 non-null  object             
 1   patient_id           57915 non-null  object             
 2   start_date           57915 non-null  datetime64[ns, UTC]
 3   end_date             57915 non-null  datetime64[ns, UTC]
 4   length_of_stay       57915 non-null  int64              
 5   disease              29628 non-null  object             
 6   procedure            57915 non-null  object             
 7   cost                 5803 non-null   float64            
 8   gender               57915 non-null  int64              
 9   age_at_episode       57915 non-null  int64              
 10  encounter_type_emer  57915 non-null  bool               
 11  encounter_type_hh    57915 non-null  bool               
 12  encounter_type_imp

In [850]:
df.describe()


,length_of_stay,cost,gender,age_at_episode
count,57915.000000,5803.0,57915.000000,57915.000000
mean,2.329189,0.0,0.512320,48.204058
std,10.913359,0.0,0.499853,21.926742
min,0.000000,0.0,0.000000,-1.000000
25%,0.000000,0.0,0.000000,31.000000
50%,0.000000,0.0,1.000000,51.000000
75%,0.000000,0.0,1.000000,65.000000
max,100.000000,0.0,1.000000,105.000000


In [852]:
df.isna().sum()


episode_id                 0
patient_id                 0
start_date                 0
end_date                   0
length_of_stay             0
disease                28287
procedure                  0
cost                   52112
gender                     0
age_at_episode             0
encounter_type_emer        0
encounter_type_hh          0
encounter_type_imp         0
encounter_type_vr          0
dtype: int64

In [854]:
df = df[df["disease"].notna()]
df = df[df["disease"].str.strip() != ""]


In [856]:
df["disease"].isna().sum(), (df["disease"].str.strip() == "").sum()


(0, 0)

In [858]:
df.isna().sum()


episode_id                 0
patient_id                 0
start_date                 0
end_date                   0
length_of_stay             0
disease                    0
procedure                  0
cost                   26744
gender                     0
age_at_episode             0
encounter_type_emer        0
encounter_type_hh          0
encounter_type_imp         0
encounter_type_vr          0
dtype: int64

In [860]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 29628 entries, 0 to 57914
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   episode_id           29628 non-null  object             
 1   patient_id           29628 non-null  object             
 2   start_date           29628 non-null  datetime64[ns, UTC]
 3   end_date             29628 non-null  datetime64[ns, UTC]
 4   length_of_stay       29628 non-null  int64              
 5   disease              29628 non-null  object             
 6   procedure            29628 non-null  object             
 7   cost                 2884 non-null   float64            
 8   gender               29628 non-null  int64              
 9   age_at_episode       29628 non-null  int64              
 10  encounter_type_emer  29628 non-null  bool               
 11  encounter_type_hh    29628 non-null  bool               
 12  encounter_type_imp   29

In [862]:
disease_counts = df["disease"].value_counts()
disease_counts


disease
medication review due (situation)           8792
gingival disease (disorder)                 2662
normal pregnancy (finding)                  1736
full-time employment (finding)              1676
stress (finding)                            1619
                                            ... 
tricuspid valve regurgitation (disorder)       1
protracted diarrhea (finding)                  1
wheezing (finding)                             1
rupture of patellar tendon (disorder)          1
constipation (finding)                         1
Name: count, Length: 212, dtype: int64

In [864]:
df["disease"].nunique()

212

In [866]:
df["disease"].value_counts().head(30)


disease
medication review due (situation)                                         8792
gingival disease (disorder)                                               2662
normal pregnancy (finding)                                                1736
full-time employment (finding)                                            1676
stress (finding)                                                          1619
primary dental caries (disorder)                                          1212
part-time employment (finding)                                            1012
gingivitis (disorder)                                                      968
infection of tooth (disorder)                                              691
social isolation (finding)                                                 670
limited social contact (finding)                                           612
not in labor force (finding)                                               587
victim of intimate partner abuse (finding)  

In [879]:
df = df[df["disease"].str.contains(r"\(disorder\)", case=False, regex=True)]


In [881]:
df["disease"].value_counts().head(20)


disease
gingival disease (disorder)                                    2662
primary dental caries (disorder)                               1212
gingivitis (disorder)                                           968
infection of tooth (disorder)                                   691
viral sinusitis (disorder)                                      335
anemia (disorder)                                               259
acute viral pharyngitis (disorder)                              234
acute bronchitis (disorder)                                     211
ischemic heart disease (disorder)                               151
hyperlipidemia (disorder)                                       105
essential hypertension (disorder)                                93
metabolic syndrome x (disorder)                                  83
pneumonia (disorder)                                             80
osteoporosis (disorder)                                          75
chronic sinusitis (disorder)            

In [883]:
df["disease_clean"] = (
    df["disease"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s*\(disorder\)", "", regex=True)
)


In [885]:
df["disease_clean"] = (
    df["disease"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s*\(.*?\)", "", regex=True)
)


In [887]:
def map_disease(d):
    if pd.isna(d):
        return None

    # Diabetes related
    if "type 2 diabetes" in d or "diabetes mellitus type 2" in d:
        return "type 2 diabetes mellitus"
    if "prediabetes" in d:
        return "diabetes related"

    # Cardiovascular
    if "hypertension" in d:
        return "hypertension"
    if "ischemic heart" in d or "coronary" in d or "heart failure" in d:
        return "cardiovascular disease"
    if "stroke" in d:
        return "cardiovascular disease"

    # Respiratory
    if "asthma" in d:
        return "asthma"
    if "viral sinusitis" in d or "acute bronchitis" in d or "viral pharyngitis" in d:
        return "respiratory infection"
    if "pneumonia" in d:
        return "respiratory infection"

    # Oral / dental
    if (
        "gingival disease" in d or
        "gingivitis" in d or
        "dental caries" in d or
        "infection of tooth" in d
    ):
        return "oral disease"

    # Urinary
    if "urinary tract" in d or "uti" in d:
        return "urinary tract infection"

    # Blood / metabolic
    if "anemia" in d:
        return "anemia"
    if "obesity" in d:
        return "metabolic disorder"

    return None


In [889]:
df["disease_group"] = df["disease_clean"].apply(map_disease)
df = df[df["disease_group"].notna()]


In [891]:
df["disease_group"].value_counts()


disease_group
oral disease                5533
respiratory infection        860
anemia                       259
cardiovascular disease       181
type 2 diabetes mellitus     165
hypertension                  93
urinary tract infection       22
asthma                         5
Name: count, dtype: int64

In [893]:
df["disease_group"].nunique()


8

In [895]:
df.groupby("disease_group")["cost"].describe()


,count,mean,std,min,25%,50%,75%,max
disease_group,,,,,,,,
anemia,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
asthma,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cardiovascular disease,28.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hypertension,44.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
oral disease,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
respiratory infection,528.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
type 2 diabetes mellitus,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
urinary tract infection,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [899]:
df["disease_group"].nunique()


8

In [901]:
def map_disease(d):
    if pd.isna(d):
        return None

    # HEART DISEASE (patient-facing)
    if (
        "ischemic heart" in d or
        "coronary" in d or
        "heart disease" in d or
        "heart failure" in d or
        "bypass graft" in d
    ):
        return "heart disease"

    # Hypertension (separate, very common)
    if "hypertension" in d:
        return "hypertension"

    # Diabetes
    if "type 2 diabetes" in d or "diabetes mellitus type 2" in d:
        return "type 2 diabetes mellitus"
    if "prediabetes" in d:
        return "diabetes related"

    # Respiratory
    if "asthma" in d:
        return "asthma"
    if (
        "viral sinusitis" in d or
        "acute bronchitis" in d or
        "viral pharyngitis" in d or
        "pneumonia" in d
    ):
        return "respiratory infection"

    # Oral / dental
    if (
        "gingival disease" in d or
        "gingivitis" in d or
        "dental caries" in d or
        "infection of tooth" in d
    ):
        return "oral disease"

    # Urinary
    if "urinary tract" in d or "uti" in d:
        return "urinary tract infection"

    # Blood / metabolic
    if "anemia" in d:
        return "anemia"
    if "obesity" in d:
        return "metabolic disorder"

    return None


In [903]:
df["disease_group"] = df["disease_clean"].apply(map_disease)
df = df[df["disease_group"].notna()]


In [905]:
df["disease_group"].value_counts()


disease_group
oral disease                5533
respiratory infection        860
anemia                       259
heart disease                181
type 2 diabetes mellitus     165
hypertension                  93
urinary tract infection       22
asthma                         5
Name: count, dtype: int64

In [907]:
df["procedure"].value_counts().head(20)


procedure
no_procedure                                                                                                   694
oral health education (procedure)                                                                              452
dental care (regime/therapy)                                                                                   452
removal of supragingival plaque and calculus from all teeth using dental instrument (procedure)                452
removal of subgingival plaque and calculus from all teeth using dental instrument (procedure)                  452
examination of gingivae (procedure)                                                                            452
dental consultation and report (procedure)                                                                     452
dental surgical procedure (procedure)                                                                          305
dental application of desensitizing medicament (procedure)            

In [909]:
pd.crosstab(df["disease_group"], df["procedure"])


procedure,antibiotic therapy (procedure),anticipatory guidance (procedure),application of composite dental filling material to dentin of tooth following fracture of tooth (procedure),assessment of anxiety (procedure),assessment of health and social care needs (procedure),assessment of substance use (procedure),assessment using alcohol use disorders identification test - consumption (procedure),"assessment using car, relax, alone, forget, friends, trouble screening test (procedure)",assessment using morse fall scale (procedure),assessment using new york heart association classification (procedure),...,syphilis infectious titer test (procedure),taking patient vital signs (procedure),throat culture (procedure),triage: emergency center (procedure),ultrasound scan for fetal viability (procedure),urinalysis with reflex to microscopy and culture (procedure),"urinalysis, protein, qualitative (procedure)",urine culture (procedure),urine screening test for diabetes (procedure),urine specimen collection (procedure)
disease_group,,,,,,,,,,,,,,,,,,,,,
anemia,0,0,0,5,15,7,4,0,0,0,...,6,0,0,0,6,0,6,6,6,0
asthma,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
heart disease,0,0,0,9,21,13,6,0,8,11,...,0,0,0,0,0,0,0,0,0,0
hypertension,0,0,0,5,8,5,4,0,0,0,...,0,0,0,0,0,0,0,0,0,0
oral disease,0,18,106,75,121,92,37,18,26,0,...,0,0,0,0,0,0,0,0,0,0
respiratory infection,0,0,0,0,0,0,0,0,0,0,...,0,0,81,0,0,0,0,0,0,0
type 2 diabetes mellitus,0,0,0,10,23,15,8,0,2,0,...,0,0,0,0,0,0,0,0,0,0
urinary tract infection,1,0,0,0,0,0,0,0,0,0,...,0,1,0,1,0,2,0,0,0,1


In [911]:
df.groupby(["disease_group", "procedure"]).size().reset_index(name="count") \
  .sort_values(["disease_group", "count"], ascending=[True, False])


,disease_group,procedure,count
23,anemia,no_procedure,56
11,anemia,depression screening (procedure),20
1,anemia,assessment of health and social care needs (pr...,15
22,anemia,medication reconciliation (procedure),8
2,anemia,assessment of substance use (procedure),7
...,...,...,...
130,urinary tract infection,history and physical examination with evaluati...,1
132,urinary tract infection,patient discharge (procedure),1
133,urinary tract infection,taking patient vital signs (procedure),1
134,urinary tract infection,triage: emergency center (procedure),1


In [913]:
proc_df = df[df["procedure"] != "no_procedure"]


In [915]:
top_procedures = (
    proc_df
    .groupby(["disease_group", "procedure"])
    .size()
    .reset_index(name="count")
    .sort_values(["disease_group", "count"], ascending=[True, False])
)


In [917]:
df[df["disease_group"] == "heart disease"].shape

(181, 16)

In [919]:
df[
    (df["disease_group"] == "heart disease") &
    (df["procedure"] != "no_procedure")
].shape


(153, 16)

In [921]:
df[
    (df["disease_group"] == "heart disease") &
    (df["procedure"] != "no_procedure")
]["procedure"].nunique()


17

In [923]:
df[
    (df["disease_group"] == "heart disease") &
    (df["procedure"] != "no_procedure")
]["procedure"].value_counts()


procedure
depression screening (procedure)                                                        30
assessment of health and social care needs (procedure)                                  21
assessment of substance use (procedure)                                                 13
patient referral for dental care (procedure)                                            12
electrocardiographic procedure (procedure)                                              11
assessment using new york heart association classification (procedure)                  11
medication reconciliation (procedure)                                                   10
assessment of anxiety (procedure)                                                        9
assessment using morse fall scale (procedure)                                            8
screening for drug abuse (procedure)                                                     7
assessment using alcohol use disorders identification test - consumption (proced

In [925]:
def map_procedure_user(p):
    if pd.isna(p):
        return None

    p = p.lower()

    if "electrocardiographic" in p:
        return "ecg"
    if "stress test" in p:
        return "cardiac stress test"
    if "new york heart association" in p:
        return "heart function assessment"
    if "x-ray of chest" in p:
        return "chest x-ray"
    if "spirometry" in p:
        return "lung function test"
    if "sleep apnea" in p:
        return "sleep apnea test"

    if "blood glucose" in p:
        return "blood glucose test"
    if "hba1c" in p:
        return "hba1c test"
    if "eye examination" in p:
        return "eye examination"
    if "foot examination" in p:
        return "foot examination"

    if "blood pressure" in p:
        return "blood pressure measurement"

    if "dental" in p or "tooth" in p:
        return "dental procedure"

    if "urine" in p:
        return "urine test"

    if "blood test" in p or "laboratory" in p:
        return "blood test"

    return None


In [928]:
df["procedure_user"] = df["procedure"].apply(map_procedure_user)


In [930]:
df.groupby("disease_group")["procedure_user"].value_counts()


disease_group             procedure_user           
anemia                    urine test                     12
                          dental procedure               11
heart disease             dental procedure               12
                          ecg                            11
                          heart function assessment      11
                          cardiac stress test             4
                          chest x-ray                     4
                          lung function test              1
                          sleep apnea test                1
hypertension              dental procedure                6
                          lung function test              1
oral disease              dental procedure             3847
                          sleep apnea test               12
                          lung function test              3
respiratory infection     chest x-ray                    15
                          dental procedure      

In [932]:
df["had_procedure"] = df["procedure_user"].notna().astype(int)


In [934]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7118 entries, 7 to 57864
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   episode_id           7118 non-null   object             
 1   patient_id           7118 non-null   object             
 2   start_date           7118 non-null   datetime64[ns, UTC]
 3   end_date             7118 non-null   datetime64[ns, UTC]
 4   length_of_stay       7118 non-null   int64              
 5   disease              7118 non-null   object             
 6   procedure            7118 non-null   object             
 7   cost                 694 non-null    float64            
 8   gender               7118 non-null   int64              
 9   age_at_episode       7118 non-null   int64              
 10  encounter_type_emer  7118 non-null   bool               
 11  encounter_type_hh    7118 non-null   bool               
 12  encounter_type_imp   711

In [936]:
df["cost"].isna().value_counts()


cost
True     6424
False     694
Name: count, dtype: int64

In [938]:
df.groupby("had_procedure")["cost"].apply(lambda x: x.isna().sum())


had_procedure
0    2453
1    3971
Name: cost, dtype: int64

In [940]:
df[
    (df["had_procedure"] == 1) &
    (df["cost"].isna())
]


,episode_id,patient_id,start_date,end_date,length_of_stay,disease,procedure,cost,gender,age_at_episode,encounter_type_emer,encounter_type_hh,encounter_type_imp,encounter_type_vr,disease_clean,disease_group,procedure_user,had_procedure
32,eb62cff7-cbc0-ce6d-c31a-2053c4274626,eb62cff7-cbc0-ce6d-f347-838afdc237d6,2017-11-12 21:27:18+00:00,2017-11-13 03:59:39+00:00,0,gingival disease (disorder),dental consultation and report (procedure),NaN,0,63,False,False,False,False,gingival disease,oral disease,dental procedure,1
33,eb62cff7-cbc0-ce6d-c31a-2053c4274626,eb62cff7-cbc0-ce6d-f347-838afdc237d6,2017-11-12 21:27:18+00:00,2017-11-13 03:59:39+00:00,0,gingival disease (disorder),dental care (regime/therapy),NaN,0,63,False,False,False,False,gingival disease,oral disease,dental procedure,1
34,eb62cff7-cbc0-ce6d-c31a-2053c4274626,eb62cff7-cbc0-ce6d-f347-838afdc237d6,2017-11-12 21:27:18+00:00,2017-11-13 03:59:39+00:00,0,gingival disease (disorder),removal of supragingival plaque and calculus f...,NaN,0,63,False,False,False,False,gingival disease,oral disease,dental procedure,1
35,eb62cff7-cbc0-ce6d-c31a-2053c4274626,eb62cff7-cbc0-ce6d-f347-838afdc237d6,2017-11-12 21:27:18+00:00,2017-11-13 03:59:39+00:00,0,gingival disease (disorder),removal of subgingival plaque and calculus fro...,NaN,0,63,False,False,False,False,gingival disease,oral disease,dental procedure,1
36,eb62cff7-cbc0-ce6d-c31a-2053c4274626,eb62cff7-cbc0-ce6d-f347-838afdc237d6,2017-11-12 21:27:18+00:00,2017-11-13 03:59:39+00:00,0,gingival disease (disorder),dental plain x-ray bitewing (procedure),NaN,0,63,False,False,False,False,gingival disease,oral disease,dental procedure,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57730,19185f7d-ec67-759c-2cc9-b888f59ca070,19185f7d-ec67-759c-f47f-1dee0aa3b385,1993-12-24 01:00:14+00:00,1993-12-24 04:28:29+00:00,0,primary dental caries (disorder),dental consultation and report (procedure),NaN,0,64,False,False,False,False,primary dental caries,oral disease,dental procedure,1
57731,19185f7d-ec67-759c-2cc9-b888f59ca070,19185f7d-ec67-759c-f47f-1dee0aa3b385,1993-12-24 01:00:14+00:00,1993-12-24 04:28:29+00:00,0,primary dental caries (disorder),dental care (regime/therapy),NaN,0,64,False,False,False,False,primary dental caries,oral disease,dental procedure,1
57732,19185f7d-ec67-759c-2cc9-b888f59ca070,19185f7d-ec67-759c-f47f-1dee0aa3b385,1993-12-24 01:00:14+00:00,1993-12-24 04:28:29+00:00,0,primary dental caries (disorder),removal of supragingival plaque and calculus f...,NaN,0,64,False,False,False,False,primary dental caries,oral disease,dental procedure,1
57733,19185f7d-ec67-759c-2cc9-b888f59ca070,19185f7d-ec67-759c-f47f-1dee0aa3b385,1993-12-24 01:00:14+00:00,1993-12-24 04:28:29+00:00,0,primary dental caries (disorder),removal of subgingival plaque and calculus fro...,NaN,0,64,False,False,False,False,primary dental caries,oral disease,dental procedure,1


In [942]:
df.loc[df["had_procedure"] == 0, "cost"] = df.loc[df["had_procedure"] == 0, "cost"].fillna(0)


In [944]:
df.loc[
    (df["had_procedure"] == 1) & (df["cost"].isna()),
    "cost"
] = df["cost"].median()


In [946]:
df["cost"].isna().sum()


0

In [948]:
df.groupby("disease_group")["cost"].describe()


,count,mean,std,min,25%,50%,75%,max
disease_group,,,,,,,,
anemia,259.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
asthma,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
heart disease,181.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hypertension,93.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
oral disease,5533.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
respiratory infection,860.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
type 2 diabetes mellitus,165.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
urinary tract infection,22.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
